### Chapter 3.4 - Linear Regression Implementation from Scratch

This notebook implements linear regression training directly with tensors. The goal is to see every moving part before using concise framework shortcuts.


In [ ]:
import math
import random
import numpy as np
import torch


In [ ]:
def data_iter(X, y, batch_size):
    indices = list(range(len(y)))
    random.shuffle(indices)
    for start in range(0, len(y), batch_size):
        idx = indices[start:start + batch_size]
        yield X[idx], y[idx]


### 3.4.1 Defining the Model

#### 1. Intuition

Defining the model means writing the function that maps input features to predictions.

For linear regression, the model is a weighted sum plus a bias: `X @ w + b`.

#### 2. Why this exists

Training cannot begin until the model's forward computation is clear. The forward computation is the path from inputs to predictions.


#### 3. Examples

Manual Python prediction for one example.


In [ ]:
x = [1.0, 2.0]
w = [0.1, -0.2]
b = 0.0
pred = x[0] * w[0] + x[1] * w[1] + b
pred


Tensor function for a batch of examples.


In [ ]:
def linreg(X, w, b):
    return X @ w + b

X = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
w = torch.tensor([0.1, -0.2])
b = torch.tensor(0.0)
linreg(X, w, b)


#### 4. Step-by-step breakdown

The manual code computes one weighted sum.

`linreg` generalizes the same formula to a batch.

`X @ w` gives one prediction per row of `X`.

`+ b` broadcasts the scalar bias to every prediction.

#### 5. Connection to ML systems

This model function is the from-scratch version of what `torch.nn.Linear` will later do internally.

#### 6. Common confusion points

- The model function should not compute loss.
- Weights must match the number of input features.
- The output shape should match the label shape.
- Initialization values are not learned yet; they are just starting points.


### 3.4.2 Defining the Loss Function

#### 1. Intuition

A loss function compares predictions with labels and returns a number representing error.

Squared loss is `(prediction - label)^2`, often averaged across a batch.

#### 2. Why this exists

The loss defines what training tries to reduce. Different losses mean different learning goals.


#### 3. Examples

Manual squared loss for two predictions.


In [ ]:
preds = [1.0, 3.0]
labels = [2.0, 1.0]
losses = [(preds[i] - labels[i]) ** 2 for i in range(2)]
sum(losses) / len(losses)


Tensor squared loss.


In [ ]:
def squared_loss(y_hat, y):
    return (y_hat - y) ** 2 / 2

y_hat = torch.tensor([1.0, 3.0])
y = torch.tensor([2.0, 1.0])
squared_loss(y_hat, y)


#### 4. Step-by-step breakdown

The manual version computes one squared error per example.

The tensor version returns per-example losses.

The division by 2 is a convenience because it cancels a factor of 2 when differentiating the square.

A training loop usually reduces these losses with `.mean()` or `.sum()`.

#### 5. Connection to ML systems

PyTorch loss modules often combine per-example computation and reduction. Writing it manually first makes the reduction choice visible.

#### 6. Common confusion points

- Per-example losses are not the same as one scalar loss.
- Dividing by 2 changes scale, not the location of the minimum.
- The loss function should compare aligned predictions and labels.
- Shape mismatch can accidentally trigger broadcasting.


### 3.4.3 Defining the Optimization Algorithm

#### 1. Intuition

An optimization algorithm updates parameters to reduce loss.

Stochastic gradient descent, or SGD, updates each parameter by moving it opposite its gradient. A gradient is the slope of the loss with respect to a parameter.

#### 2. Why this exists

The model learns only when parameters change. Gradients tell us direction. The learning rate controls step size.


#### 3. Examples

Manual SGD update for one scalar parameter.


In [ ]:
w = 3.0
grad = 4.0
lr = 0.1
new_w = w - lr * grad
new_w


Tensor SGD update for a list of parameters.


In [ ]:
def sgd(params, lr, batch_size):
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad / batch_size
            p.grad.zero_()


#### 4. Step-by-step breakdown

`new_w = w - lr * grad` moves the parameter opposite the gradient.

`with torch.no_grad()` prevents PyTorch from tracking the update itself.

The loop visits each parameter tensor.

`p.grad.zero_()` clears old gradients so the next backward pass starts fresh.

Dividing by `batch_size` keeps update scale consistent when the loss was summed.

#### 5. Connection to ML systems

Framework optimizers automate this same pattern. They store parameter references and update them after `backward()` computes gradients.

#### 6. Common confusion points

- Optimizers update parameters, not predictions directly.
- Learning rate too large can make training unstable.
- Gradients accumulate unless cleared.
- Parameter updates should usually happen inside `torch.no_grad()`.


### 3.4.4 Training

#### 1. Intuition

Training repeats a sequence: get a batch, predict, compute loss, compute gradients, update parameters.

What changes each iteration are the parameter values and the current batch.

#### 2. Why this exists

The training loop turns model definition, loss, and optimization into learning. Without the loop, the pieces do not improve.


#### 3. Examples

A compact from-scratch training loop on synthetic data.


In [ ]:
X = torch.randn(20, 2)
y = X @ torch.tensor([2.0, -3.4]) + 4.2 + torch.randn(20) * 0.01
w = torch.randn(2, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
for epoch in range(3):
    for batch_X, batch_y in data_iter(X, y, 5):
        loss = squared_loss(linreg(batch_X, w, b), batch_y).sum()
        loss.backward()
        sgd([w, b], lr=0.03, batch_size=5)
w, b


#### 4. Step-by-step breakdown

The true parameters create synthetic labels.

`w` and `b` are learnable parameters because they have `requires_grad=True`.

The outer loop repeats epochs.

The inner loop reads one minibatch at a time.

For each minibatch, the code computes loss, calls `backward()`, and updates parameters with SGD.

#### 5. Connection to ML systems

This is the essential PyTorch training loop without optimizer classes or module abstractions. Later concise code compresses the same sequence.

#### 6. Common confusion points

- `backward()` must happen before the SGD update.
- Gradients must be cleared after each update.
- The current batch changes inside the inner loop.
- Learned parameters should move toward true parameters on this synthetic task, but noise prevents exact equality.


### 3.4.5 Summary

#### 1. Intuition

From-scratch linear regression has four core parts: model, loss, optimizer, and training loop.

#### 2. Why this exists

Writing these parts manually builds the mental model needed to understand framework abstractions.


#### 3. Examples

The training sequence as explicit names.


In [ ]:
steps = [
    "read batch",
    "predict",
    "compute loss",
    "backward",
    "update parameters",
]
steps


#### 4. Step-by-step breakdown

The list records the order of operations.

Every training iteration follows this order.

If the order changes accidentally, training may fail or silently behave incorrectly.

#### 5. Connection to ML systems

Most deep learning systems are variations of this same sequence, even when wrapped in classes and library calls.

#### 6. Common confusion points

- A clear training loop is more important than a clever one.
- Loss scale affects gradient scale.
- Parameter initialization affects the starting point.
- Debug from-scratch code with tiny synthetic data first.


### 3.4.6 Exercises

#### 1. Intuition

These exercises make you inspect and modify the training loop.

#### 2. Why this exists

Small changes reveal which parts of training control speed, stability, and correctness.


#### 3. Examples

Exercise 1: run one prediction with current parameters.


In [ ]:
sample_X = torch.tensor([[1.0, 2.0]])
linreg(sample_X, w.detach(), b.detach())


Exercise 2: compute one batch loss without updating.


In [ ]:
batch_X, batch_y = next(data_iter(X, y, 5))
y_hat = linreg(batch_X, w.detach(), b.detach())
squared_loss(y_hat, batch_y).mean()


#### 4. Step-by-step breakdown

Exercise 1 checks the model function after training.

Exercise 2 separates evaluation from updating. `detach()` gives values without tracking gradients.

This separation helps debug whether prediction and loss work before training changes anything.

#### 5. Connection to ML systems

Evaluation code should usually avoid building gradient graphs because it does not update parameters.

#### 6. Common confusion points

- `detach()` stops gradient tracking for that value.
- Evaluation loss is not a parameter update.
- A single batch loss is noisy.
- Always know whether your code is training or only measuring.
